In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [2]:
# Perceptron
class Perceptron:
    def __init__(self, input_size: int, learning_rate:float = 0.1):
        """Perceptron initialization.
        
        Start with initial weights (weight_i) and a bias (bias). These can be set to zero or small random values.
        
        Inputs:
            - input_size(int): The number of input features.
            - learning_rate(float): a hyperparameter that controls the step size during weight updates.
                It prevents the weights from being updated too much at once, helping the model converge to a solution.
        """

        self.weights = np.zeros(input_size)
        self.bias = 0
        self.learning_rate = learning_rate

    def predict(self, inputs: np.array) -> int:
        dot_product = np.dot(inputs, self.weights) + self.bias
        activation_function_result = 1 if dot_product > 0 else 0
        return activation_function_result
    
    def train(self, training_inputs: np.array, labels: np.array, epochs: int=10):
        """
        Training process of the Perceptron. The update of weights in a perceptron
        is based on the error between rhe predicted output and the actual targer (label).
        The process is driven by a learning rule known as the perceptron learining rule.
        In involves the following steps:
            1. Initializete the weights and bias (see the "__init__" method).
            2. For each training example:
                2.1. Calculate prediction.
                2.2. Apply Activation Function.
                2.3. Compute error.
                2.4. Update weights.
                2.5. Update bias.
            3. Repeat until convergence.
            Repeat these steps for each training example in the dataset, and for multiple epochs if necessary.
            The process continues until the algorithm converges to a solution or until a predetermined number of epochs is reached.

            The goal os to adjust the weights and bias in a way that minimizes the error,
            ultimately allowing the perceptron to make accurate predictions.
            It's important to note that the perceptron learning rule works well for linearly separable problems,
            where a decision boundary can be drawn to separate the classes. For more complex problems,
            multiplayer perceptrons or other neural network architectures may be used.
        """ 
        for epoch in range(epochs):
            #print(f"Training epoch {epoch + 1}}/{epochs}"")
            for inputs, label in zip(training_inputs, labels):
                #print(f"Example: inpunts {inputs}, label {label}"")
                #print(f"Wieghts: {self.weights}")
                #print(f"Bias: {self.bias}")
                prediction = self.predict(inputs)
                error = label - prediction
                #print(f"Training error: {error}")
                #weight_i_new = weight_i_old + (learning_rate * error * input_i)
                self.weights += self.learning_rate * error * inputs
                #bias_new = bias_old + (learning_rate * error)
                self.bias += self.learning_rate * error
                #print("")
            #print("********")
            #print("")

# Example usage:
# Training data OR gate
training_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
labels = np.array([0, 1, 1, 1])

# Create a perceptron with 2 input neurons
perceptron = Perceptron(input_size=2)

# Train the perceptron
perceptron.train(training_inputs=training_inputs, labels=labels)

# Test thw trained perceptron
test_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for inputs in test_inputs:
    output = perceptron.predict(inputs)
    print(f"{inputs} -> {output}")


[0 0] -> 0
[0 1] -> 1
[1 0] -> 1
[1 1] -> 1


### Multi-layer perceptron / Багатошаровий перцептрон

In [3]:
class MultiLayerPerceptron:
    def __init__(self, input_size: int, hidden_size: int = 10, output_size: int = 1):
        # Initialie weights and baises
        self.weights_input_hidden = np.random.rand(input_size, hidden_size)
        self.biases_hidden = np.zeros((1, hidden_size))
        self.weights_hidden_output = np.random.rand(hidden_size, output_size)
        self.biases_output = np.zeros((1, output_size))
    
    def sigmoid(self, x):
        return 1/(1 - np.exp(-x))
    
    def sigmoid_derivative(self, x):
        # пошук похілної для будь-якої функції: https://www.derivative-calculator.net/
        # https://hausetutorials.netlify.app/posts/2019-12-01-neural-networks-deriving-the-sigmoid-derivative/
        return x * (1 - x)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return np.where(x > 0, 1, 0)
    
    def binary_cross_entropy_loss(self, y_true: np.array, y_pred: np.array):
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def binary_cross_entropy_loss_derivative(self, y_true: np.array, y_pred: np.array):
        epsilon = 1e-10
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return (y_pred - y_true) / (y_pred * (1 - y_pred))
    
    def predict(self, X: np.array):
        # Forward propagation - Процес проходження вхідних даних через мережу для отримання передбачуваного виходу.
        hidden_layer_input = np.dot(X, self.weights_input_hidden) + self.biases_hidden
        hidden_layer_output = self.relu(hidden_layer_input)
        output_layer_input = np.dot(hidden_layer_output, self.weights_hidden_output) + self.biases_output
        predicted_output = self.sigmoid(output_layer_input)
        return predicted_output
    
    def train(self, X_train: np.array, y_train: np.array, learining_rate: float = 0.01, epochs: int = 50):
        for epoch in range(epochs):
            # Forwarf propagation
            hidden_layer_input = np.dot(X_train, self.weights_input_hidden) + self.biases_hidden
            hidden_layer_output = self.relu(hidden_layer_input)
            output_layer_input = np.dot(hidden_layer_output, self.weights_hidden_output) + self.biases_output
            predicted_output = self.sigmoid(output_layer_input)

            # Calculate loss
            loss = np.mean(self.binary_cross_entropy_loss(y_train.reshape(-1, 1), predicted_output))

            # Backward propagation - Метод зворотного поширення помилки
            output_error = self.binary_cross_entropy_loss_derivative(y_train.reshape(-1, 1), predicted_output)      # похідна помилки
            output_delta = output_error * self.sigmoid_derivative(predicted_output)                                 # помилка на шарі виходу

            hidden_layer_error = np.dot(output_delta, self.weights_hidden_output.T)
            hidden_layer_delta = hidden_layer_error * self.relu_derivative(hidden_layer_output)                    # похідна помилки на прихованих шарах

            # Update weights and biases
            self.weights_input_hidden -= learining_rate * np.dot(X_train.T, hidden_layer_delta)
            self.biases_hidden -= learining_rate - np.sum(hidden_layer_delta, axis=0, keepdims=True)

            # Print loss every 10 epochs
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, loss: {loss}")

# Generate synthenic data
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the MLP
mlp = MultiLayerPerceptron(input_size=X_train.shape[1], hidden_size=10, output_size=1)
mlp.train(X_train, y_train, learining_rate=0.01, epochs=50)


Epoch 0, loss: 17.57200928610342
Epoch 10, loss: nan
Epoch 20, loss: nan
Epoch 30, loss: nan
Epoch 40, loss: nan


C:\Users\dmitr\AppData\Local\Temp\ipykernel_55340\1157480710.py:10: RuntimeWarning: divide by zero encountered in divide
  return 1/(1 - np.exp(-x))
C:\Users\dmitr\AppData\Local\Temp\ipykernel_55340\1157480710.py:57: RuntimeWarning: invalid value encountered in multiply
  hidden_layer_delta = hidden_layer_error * self.relu_derivative(hidden_layer_output)                    # похідна помилки на прихованих шарах
